# Animation: K-Means Convergence

Watch how K-Means iterates:
1. **Assign** — each point gets the color of its nearest centroid
2. **Update** — centroids move to the mean of their cluster
3. Repeat until nothing changes

> No external packages needed — just `numpy` and `matplotlib`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

np.random.seed(7)

# ── Generate blobs ──────────────────────────────────────────────
K = 4
COLORS = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

centers_true = np.array([[1, 1], [4, 4], [1, 4], [4, 1]], dtype=float)
X = np.vstack([
    centers_true[k] + np.random.randn(60, 2) * 0.6
    for k in range(K)
])

# ── Run K-Means and record every step ───────────────────────────
centroids = X[np.random.choice(len(X), K, replace=False)].copy()
history = []  # list of (centroids, labels) per iteration

for _ in range(12):
    # Assign
    dists = np.linalg.norm(X[:, None] - centroids[None], axis=2)
    labels = np.argmin(dists, axis=1)
    history.append((centroids.copy(), labels.copy()))
    # Update
    new_centroids = np.array([X[labels == k].mean(axis=0) for k in range(K)])
    if np.allclose(new_centroids, centroids):
        history.append((new_centroids.copy(), labels.copy()))  # final state
        break
    centroids = new_centroids

print(f'K-Means converged in {len(history)} steps.')

K-Means converged in 4 steps.


In [2]:
# ── Build animation ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(-1, 6)
ax.set_ylim(-1, 6)
ax.set_aspect('equal')
ax.set_title('K-Means Convergence', fontsize=14)

scat = ax.scatter([], [], s=55, zorder=2)
cent_scat = ax.scatter([], [], s=280, marker='*', edgecolors='black',
                        linewidths=1.2, zorder=3)
step_text = ax.text(0.02, 0.97, '', transform=ax.transAxes,
                    fontsize=11, va='top', fontweight='bold')

legend_patches = [mpatches.Patch(color=COLORS[k], label=f'Cluster {k+1}')
                  for k in range(K)]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)

def update(frame):
    cents, labs = history[frame]
    point_colors = [COLORS[l] for l in labs]
    cent_colors  = COLORS[:K]

    scat.set_offsets(X)
    scat.set_color(point_colors)

    cent_scat.set_offsets(cents)
    cent_scat.set_color(cent_colors)

    status = 'Converged!' if frame == len(history) - 1 else f'Step {frame + 1}'
    step_text.set_text(status)
    return scat, cent_scat, step_text

anim = FuncAnimation(fig, update, frames=len(history),
                     interval=900, repeat=True, blit=True)
plt.close()
HTML(anim.to_jshtml())

## What to notice

- **Step 1:** Random initialization — centroids are scattered randomly across the data
- **Early steps:** Big jumps as centroids race toward cluster centers
- **Later steps:** Tiny adjustments — the algorithm is nearly converged
- **Final step:** Nothing moves — the assignment no longer changes

This is why K-Means is sensitive to initialization: a bad starting point can lead to a suboptimal solution. That's what **K-Means++** initialization solves.